# Sprint 4 — Tuning, Export TFLite & Interprétabilité

**AquaSense AI · EHTP MIG S4**

Objectifs :
- Grid search (dropout, lr, batch_size)
- Weighted loss + L2 regularization
- Export `.keras` + `.tflite`
- Permutation importance vs XGBoost
- Analyse : DL sur données tabulaires

In [ ]:
# --- Configuration Google Colab ---
import sys
import os

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Ajustez le chemin vers le dossier contenant vos notebooks si besoin
    PROJECT_DRIVE_PATH = '/content/drive/MyDrive/aquasense-ai/notebooks'
    if os.path.exists(PROJECT_DRIVE_PATH):
        os.chdir(PROJECT_DRIVE_PATH)
        print(f"Colab : Dossier courant changé vers {os.getcwd()}")
    else:
        print(f"Colab : Dossier {PROJECT_DRIVE_PATH} introuvable. Veuillez ajuster le chemin.")
else:
    print("Environnement local détecté.")
# ----------------------------------


In [ ]:
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.dl_utils import (
    MODELS_DIR,
    REPORTS_DIR,
    build_mlp,
    compile_model,
    evaluate_keras_model,
    export_tflite,
    load_ml_comparison_metrics,
    permutation_importance_keras,
    prepare_dl_data,
    train_keras_model,
)
from src.train import get_encoded_feature_names, NEEDS_REPAIR

sns.set_theme(style="whitegrid")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
EPOCHS = 100

## 1. Préparation des données

In [ ]:
data = prepare_dl_data(test_size=0.2)

## 2. Grid search

Hyperparamètres : dropout ∈ {0.2, 0.3, 0.5}, lr ∈ {1e-2, 1e-3, 1e-4}, batch_size ∈ {256, 512, 1024}.

Loss pondérée (inverse des fréquences) + L2(0.001).

In [ ]:
grid_results = []
best_score = -1.0
best_model = None
best_cfg = {}

for dropout in [0.2, 0.3, 0.5]:
    for lr in [1e-2, 1e-3, 1e-4]:
        for batch_size in [256, 512, 1024]:
            model = build_mlp(data["input_dim"], dropout=dropout, l2_reg=0.001)
            compile_model(
                model,
                lr=lr,
                class_weight=data["class_weight"],
                use_weighted_loss=True,
            )
            train_keras_model(
                model,
                data,
                epochs=EPOCHS,
                batch_size=batch_size,
                validation_split=None,
                use_class_weight=False,
                use_sample_weight=True,
                verbose=0,
            )
            metrics = evaluate_keras_model(model, data["X_val"], data["y_val"], data["label_encoder"])
            row = {
                "dropout": dropout,
                "lr": lr,
                "batch_size": batch_size,
                "f1_macro": metrics["f1_macro"],
                "f1_needs_repair": metrics["f1_needs_repair"],
                "accuracy": metrics["accuracy"],
            }
            grid_results.append(row)
            if metrics["f1_macro"] > best_score:
                best_score = metrics["f1_macro"]
                best_model = model
                best_cfg = row
            print(f"d={dropout} lr={lr} bs={batch_size} → F1={metrics['f1_macro']:.4f}")

grid_df = pd.DataFrame(grid_results).sort_values("f1_macro", ascending=False)
grid_df.head(10)

## 3. L2 regularization : avec vs sans

In [ ]:
l2_results = []
for l2 in [0.0, 0.001]:
    model = build_mlp(data["input_dim"], dropout=0.3, l2_reg=l2)
    compile_model(model, lr=1e-3, class_weight=data["class_weight"])
    train_keras_model(model, data, epochs=EPOCHS, batch_size=512, validation_split=None, verbose=0)
    m = evaluate_keras_model(model, data["X_val"], data["y_val"], data["label_encoder"])
    l2_results.append({"l2_reg": l2, **m})

pd.DataFrame(l2_results)[["l2_reg", "f1_macro", "f1_needs_repair", "accuracy"]]

## 4. Sauvegarde meilleur modèle + export TFLite

In [ ]:
keras_path = MODELS_DIR / "mlp_tuned_best_v1.keras"
tflite_path = MODELS_DIR / "mlp_tuned_best_v1.tflite"

best_model.save(keras_path)
export_tflite(best_model, tflite_path)

print("Meilleure config :", best_cfg)
print(f"Keras  → {keras_path} ({keras_path.stat().st_size / 1024:.1f} KB)")
print(f"TFLite → {tflite_path} ({tflite_path.stat().st_size / 1024:.1f} KB)")

## 5. Tableau comparatif ML vs DL

In [ ]:
ml_df = load_ml_comparison_metrics()
if not ml_df.empty:
    ml_top = ml_df.sort_values("f1_macro", ascending=False).head(3).copy()
    ml_top["type"] = "ML"
else:
    ml_top = pd.DataFrame()

dl_top = pd.DataFrame([
    {"model": "MLP tuned (best grid)", "f1_macro": best_cfg["f1_macro"], "f1_needs_repair": best_cfg["f1_needs_repair"], "accuracy": best_cfg["accuracy"], "type": "DL"},
])

ml_vs_dl = pd.concat([ml_top, dl_top], ignore_index=True)
ml_vs_dl.to_csv(REPORTS_DIR / "sprint_04_ml_vs_dl.csv", index=False)
ml_vs_dl

## 6. Permutation importance (DL) vs Feature importance (XGBoost)

In [ ]:
perm_df = permutation_importance_keras(
    best_model,
    data["X_val"][:3000],
    data["y_val"][:3000],
    data["feature_names"],
    n_repeats=5,
)
perm_top = perm_df.head(15)

xgb_path = MODELS_DIR / "xgb_best_v1.joblib"
if xgb_path.exists():
    xgb_pipe = joblib.load(xgb_path)
    prep = xgb_pipe.named_steps["prep"]
    xgb_clf = xgb_pipe.named_steps["clf"]
    feat_names = get_encoded_feature_names(prep)
    xgb_imp = pd.Series(xgb_clf.feature_importances_, index=feat_names).sort_values(ascending=False).head(15)
else:
    xgb_imp = pd.Series(dtype=float)
    print("xgb_best_v1.joblib absent — exécuter Sprint 3 d'abord.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].barh(perm_top["feature"][::-1], perm_top["importance"][::-1], color="#3498db")
axes[0].set_title("Permutation importance — MLP (DL)")
axes[0].set_xlabel("Δ F1-Macro")

if len(xgb_imp):
    axes[1].barh(xgb_imp.index[::-1], xgb_imp.values[::-1], color="#2ecc71")
    axes[1].set_title("Feature importance — XGBoost (ML)")
    axes[1].set_xlabel("Importance")

plt.tight_layout()
plt.show()

## 7. Analyse — Pourquoi le DL aide (ou pas) sur les données tabulaires

### Contexte du problème AquaSense

La prédiction de l'état des pompes à eau repose sur un jeu de données **entièrement tabulaire** : variables numériques (âge, altitude GPS, population), catégorielles (bassin, type d'extraction, financement) et features engineering géospatiales (`dist_to_basin_center`, coordonnées). Contrairement aux images ou au texte, ces données n'ont **pas de structure locale exploitable** par des convolutions : l'ordre des colonnes après encodage One-Hot est arbitraire.

### Forces du Deep Learning ici

Les réseaux profonds (MLP, Residual MLP) excellent à modéliser des **interactions non-linéaires** entre dizaines de features encodées. Le BatchNormalization stabilise l'entraînement malgré le fort déséquilibre de classes (7 % « needs repair »). La **loss pondérée** et les `class_weight` compensent partiellement la rareté de la classe critique. L'export **TFLite** ouvre la voie au déploiement embarqué sur capteurs IoT simulés (Sprint 5+), où la latence et la taille mémoire comptent.

Le Residual MLP atténue le vanishing gradient et permet des blocs plus profonds ; le 1D-CNN traite le vecteur de features comme une « séquence » artificielle — approche parfois utile quand des groupes de variables corrélées sont adjacents après encodage, mais **sans garantie théorique** sur données hétérogènes.

### Limites observées

Sur ce benchmark, **XGBoost et Random Forest** restent souvent compétitifs voire supérieurs : ils gèrent nativement les splits sur features catégorielles ordinales, sont robustes aux outliers (construction_year=0 corrigés, TSH extrêmes) et nécessitent moins de tuning. Le DL exige un préprocessing lourd (One-Hot → dimension élevée), plus de données pour généraliser, et un grid search coûteux (27 combinaisons × 100 epochs).

La **Recall « needs repair »** reste le goulot : même avec loss pondérée, le MLP peut favoriser les classes majoritaires si le signal est faible. L'**interprétabilité** est moindre : la permutation importance DL est plus coûteuse et moins stable que l'importance Gini/XGBoost, ce qui freine l'adoption par les opérateurs de maintenance terrain au Maroc.

### Conclusion

Le Deep Learning **n'apporte pas de gain systématique** sur données tabulaires structurées de ce type. Il devient pertinent si l'on enrichit le pipeline (embeddings appris pour haute cardinalité, fusion avec séries temporelles capteurs, semi-supervisé). Pour AquaSense en l'état, **l'ensemble ML (XGBoost) + règles métier** constitue la baseline à battre ; le DL sert surtout de **comparatif méthodologique**, de preuve de déployabilité TFLite, et de fondation pour des sprints futurs intégrant des signaux IoT continus où les réseaux récurrents ou 1D-CNN sur séries temporelles seraient réellement adaptés.